In [37]:
import pandas as pd
import sqlite3
import re
import numpy as np

# =========================================================
# Helpers
# =========================================================
def read_csv_safe(path):
    """Lee CSV intentando utf-8 y fallback latin-1."""
    try:
        return pd.read_csv(path)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin-1")

def pick_first(cols, etiqueta, df_columns):
    if not cols:
        raise ValueError(f"No se encontró columna para {etiqueta}. Columnas disponibles: {list(df_columns)}")
    if len(cols) > 1:
        print(f"⚠️ Varias columnas para {etiqueta}: {cols} -> usando: {cols[0]}")
    return cols[0]

def normalize_text(s: pd.Series) -> pd.Series:
    """Normaliza texto para joins (quita acentos, colapsa espacios, upper/strip)."""
    return (
        s.astype(str)
         .str.normalize("NFKD")
         .str.encode("ascii", "ignore")
         .str.decode("utf-8")
         .str.upper()
         .str.replace(r"\s+", " ", regex=True)
         .str.strip()
    )

def clean_money_to_numeric(s: pd.Series) -> pd.Series:
    """Convierte dinero texto a numérico removiendo comillas y separadores."""
    s = s.astype(str).str.replace('"', '', regex=False).str.replace(',', '', regex=False)
    return pd.to_numeric(s, errors="coerce")

def extract_loan(series: pd.Series, pattern: str) -> pd.Series:
    """Extrae número de préstamo como Series (no DataFrame)."""
    if series is None:
        return pd.Series([np.nan] * 0)
    return series.astype(str).str.extract(pattern, expand=False)

# =========================================================
# 1. EXTRACCIÓN
# =========================================================
print("Cargando CSVs...")
df_ops   = read_csv_safe("operaciones_indicadores_producto_efecto.csv")
df_adj   = read_csv_safe("adjudicatarios_psd.csv")
df_proc  = read_csv_safe("procesos_adquisiciones_psd.csv")
df_desem = read_csv_safe("desembolsos-prestamos-sector-publico.csv")
df_docs  = read_csv_safe("documentos_pai.csv")
df_solic = read_csv_safe("solicitudes_informacion_pai.csv")

# =========================================================
# 2. DIMENSIÓN OPERACIONES
# =========================================================
print("Creando Dimensión Operaciones...")
needed_ops = ["LINEA_CREDITO_NO", "NOMBRE_OPERACION", "PAIS", "SECTOR_INSTITUCIONAL", "MONTO_APROBADO"]
missing = [c for c in needed_ops if c not in df_ops.columns]
if missing:
    raise ValueError(f"Faltan columnas en df_ops: {missing}. Disponibles: {list(df_ops.columns)}")

dim_operaciones = df_ops[needed_ops].copy()
dim_operaciones.rename(columns={"LINEA_CREDITO_NO": "numero_prestamo"}, inplace=True)

dim_operaciones["numero_prestamo"] = dim_operaciones["numero_prestamo"].astype(str).str.strip()
dim_operaciones["MONTO_APROBADO"]  = pd.to_numeric(dim_operaciones["MONTO_APROBADO"], errors="coerce")

dim_operaciones.drop_duplicates(subset=["numero_prestamo"], inplace=True)

# =========================================================
# 3. HECHOS: INDICADORES DE IMPACTO (emparejando por sufijo)
# =========================================================
print("Procesando Indicadores de Impacto...")

# Detecta columnas tipo NOMBRE_INDICADOR_PRODUCTO_* y ULTIMO_VALOR_OBSERVADO_*
pairs = {}  # idx -> {"name": col, "value": col}
for c in df_ops.columns:
    m = re.search(r"(NOMBRE_INDICADOR_PRODUCTO|ULTIMO_VALOR_OBSERVADO)[\s_]*([0-9]+)", str(c))
    if m:
        kind, idx = m.group(1), m.group(2)
        pairs.setdefault(idx, {})
        if kind == "NOMBRE_INDICADOR_PRODUCTO":
            pairs[idx]["name"] = c
        else:
            pairs[idx]["value"] = c

valid_idxs = [idx for idx, d in pairs.items() if "name" in d and "value" in d]
if not valid_idxs:
    # fallback simple: por si el dataset no trae sufijos
    cols_valores = [col for col in df_ops.columns if "ULTIMO_VALOR_OBSERVADO" in col]
    cols_nombres = [col for col in df_ops.columns if "NOMBRE_INDICADOR_PRODUCTO" in col]
    min_len = min(len(cols_nombres), len(cols_valores))
    valid_pairs = list(zip(cols_nombres[:min_len], cols_valores[:min_len]))
else:
    valid_pairs = [(pairs[idx]["name"], pairs[idx]["value"]) for idx in sorted(valid_idxs, key=lambda x: int(x))]

frames = []
for nombre_col, valor_col in valid_pairs:
    tmp = df_ops[["LINEA_CREDITO_NO", nombre_col, valor_col]].copy()
    tmp.columns = ["numero_prestamo", "nombre_metrica", "valor_observado"]
    frames.append(tmp)

fact_indicadores = pd.concat(frames, ignore_index=True)

fact_indicadores["numero_prestamo"]  = fact_indicadores["numero_prestamo"].astype(str).str.strip()
fact_indicadores["valor_observado"]  = pd.to_numeric(fact_indicadores["valor_observado"], errors="coerce")
fact_indicadores.dropna(subset=["nombre_metrica"], inplace=True)
fact_indicadores = fact_indicadores[fact_indicadores["valor_observado"] > 0]

# =========================================================
# 4. HECHOS: ADJUDICACIONES (merge con procesos)
# =========================================================
print("Unificando Adjudicaciones y Procesos...")

if "IDENTIFICADOR_PROCESO" not in df_adj.columns:
    raise ValueError("df_adj no tiene IDENTIFICADOR_PROCESO.")
if "IDENTIFICADOR_PROCESO" not in df_proc.columns or "NOMBRE_ADQUISICION" not in df_proc.columns:
    raise ValueError("df_proc debe tener IDENTIFICADOR_PROCESO y NOMBRE_ADQUISICION.")

fact_adjudicaciones = pd.merge(
    df_adj,
    df_proc[["IDENTIFICADOR_PROCESO", "NOMBRE_ADQUISICION"]],
    on="IDENTIFICADOR_PROCESO",
    how="left"
)

if "NUMERO_PRESTAMO" in fact_adjudicaciones.columns:
    fact_adjudicaciones.rename(columns={"NUMERO_PRESTAMO": "numero_prestamo"}, inplace=True)

fact_adjudicaciones["numero_prestamo"] = fact_adjudicaciones["numero_prestamo"].astype(str).str.strip()
fact_adjudicaciones["monto_adjudicado_usd"] = pd.to_numeric(
    fact_adjudicaciones.get("MONTO_ADJUDICADO_DOLAR", np.nan),
    errors="coerce"
)

cols_out = ["numero_prestamo", "IDENTIFICADOR_PROCESO", "NOMBRE_ADQUISICION",
            "NOMBRE_ADJUDICATARIO", "PAIS_ADJUDICATARIO", "monto_adjudicado_usd"]
fact_adjudicaciones = fact_adjudicaciones[[c for c in cols_out if c in fact_adjudicaciones.columns]].copy()

# =========================================================
# 5. HECHOS: DESEMBOLSOS (join robusto)
# =========================================================
print("Procesando Desembolsos...")

col_proyecto_lista = [c for c in df_desem.columns if ("PROYECTO" in c.upper() or "OPERACION" in c.upper())]
col_monto_lista    = [c for c in df_desem.columns if "MONTO" in c.upper()]
col_anio_lista     = [c for c in df_desem.columns if ("AÑO" in c.upper() or "ANIO" in c.upper())]

col_proyecto = pick_first(col_proyecto_lista, "proyecto/operación", df_desem.columns)
col_monto    = pick_first(col_monto_lista, "monto", df_desem.columns)
col_anio     = pick_first(col_anio_lista, "año", df_desem.columns)

print("Columna proyecto/operación:", col_proyecto)
print("Columna monto:", col_monto)
print("Columna año:", col_anio)

df_desem["join_key"]        = normalize_text(df_desem[col_proyecto])
dim_operaciones["join_key"] = normalize_text(dim_operaciones["NOMBRE_OPERACION"])

fact_desembolsos = pd.merge(
    df_desem,
    dim_operaciones[["join_key", "numero_prestamo"]],
    on="join_key",
    how="inner"
)

fact_desembolsos.rename(columns={col_anio: "anio", col_monto: "monto_desembolsado_usd"}, inplace=True)
fact_desembolsos["monto_desembolsado_usd"] = clean_money_to_numeric(fact_desembolsos["monto_desembolsado_usd"])

fact_desembolsos = fact_desembolsos[["numero_prestamo", "anio", "monto_desembolsado_usd"]].copy()
dim_operaciones.drop(columns=["join_key"], inplace=True, errors="ignore")

# =========================================================
# 6. “NEURO-SIMBÓLICA”: REGEX TRANSPARENCIA (FIX incluido)
# =========================================================
print("Aplicando Regex para auditoría cívica...")

# Más flexible: 4 o más dígitos (ajústalo si tus préstamos son estrictamente 4)
patron_regex = r"(?i)(?:ptmo|pr[ée]stamo|no\.?|#)\s*([0-9]{4,})"

# Documentos
if "TITULO_DOCUMENTO" not in df_docs.columns:
    raise ValueError("df_docs no tiene TITULO_DOCUMENTO.")

df_docs["numero_prestamo_vinculado"] = extract_loan(df_docs["TITULO_DOCUMENTO"], patron_regex)

# Solo si existe OPERACION
if "OPERACION" in df_docs.columns:
    df_docs["numero_prestamo_vinculado"] = df_docs["numero_prestamo_vinculado"].fillna(
        extract_loan(df_docs["OPERACION"], patron_regex)
    )

# Solicitudes
if "RESUMEN_SOLICITUD" not in df_solic.columns:
    raise ValueError("df_solic no tiene RESUMEN_SOLICITUD.")

df_solic["numero_prestamo_vinculado"] = extract_loan(df_solic["RESUMEN_SOLICITUD"], patron_regex)

# Dims transparencia
needed_docs = ["CODIGO_DOCUMENTO", "TITULO_DOCUMENTO", "TIPO_DOCUMENTAL", "numero_prestamo_vinculado"]
dim_documentos = df_docs[[c for c in needed_docs if c in df_docs.columns]].dropna(subset=["numero_prestamo_vinculado"]).copy()

needed_solic = ["CODIGO_SOLICITUD", "RESUMEN_SOLICITUD", "ESTADO_SOLICITUD", "numero_prestamo_vinculado"]
dim_solicitudes = df_solic[[c for c in needed_solic if c in df_solic.columns]].dropna(subset=["numero_prestamo_vinculado"]).copy()

# =========================================================
# 7. CARGA SQLITE
# =========================================================
print("Volcando datos a SQLite (nashai.db)...")
conn = sqlite3.connect("nashai.db")

dim_operaciones.to_sql("dim_operaciones", conn, if_exists="replace", index=False)
fact_indicadores.to_sql("fact_indicadores_impacto", conn, if_exists="replace", index=False)
fact_adjudicaciones.to_sql("fact_adjudicaciones", conn, if_exists="replace", index=False)
fact_desembolsos.to_sql("fact_desembolsos", conn, if_exists="replace", index=False)
dim_documentos.to_sql("dim_transparencia_docs", conn, if_exists="replace", index=False)
dim_solicitudes.to_sql("dim_transparencia_solicitudes", conn, if_exists="replace", index=False)

conn.close()
print("¡Éxito! Base de datos 'nashai.db' creada correctamente.")

Cargando CSVs...
Creando Dimensión Operaciones...
Procesando Indicadores de Impacto...
Unificando Adjudicaciones y Procesos...
Procesando Desembolsos...
Columna proyecto/operación: DESCRIPCION_PROYECTO
Columna monto: MONTO_BRUTO_USD
Columna año: ANIO_DESEMBOLSO
Aplicando Regex para auditoría cívica...
Volcando datos a SQLite (nashai.db)...
¡Éxito! Base de datos 'nashai.db' creada correctamente.
